# ⚽ FootballFlow — Bronze Layer
## Goal: Read events from Kafka and write them as raw Parquet files

```
Kafka Topic (real_match_events)
         ↓
   Spark Streaming
         ↓
  Bronze Parquet Files
  ~/footballflow/bronze/match_events/
```

> **Bronze Rule:** No transformations are applied to the data — we store the raw JSON as-is, plus metadata columns only.

---
## Cell 1 — SparkSession
Create a SparkSession with the required JARs for Kafka connectivity.

In [1]:
import os
from pyspark.sql import SparkSession

# Stop any existing session if running
try:
    spark.stop()
    print("⏹ Old session stopped")
except:
    pass

# Path to the JARs required for Spark-Kafka integration
jar_dir = os.path.expanduser("~/kafka-jars")
jars = ",".join([
    f"{jar_dir}/spark-sql-kafka-0-10_2.12-3.5.3.jar",
    f"{jar_dir}/kafka-clients-3.4.1.jar",
    f"{jar_dir}/spark-token-provider-kafka-0-10_2.12-3.5.3.jar",
    f"{jar_dir}/commons-pool2-2.11.1.jar"
])

spark = SparkSession.builder \
    .appName("FootballFlow-Bronze") \
    .config("spark.jars", jars) \
    .config("spark.sql.shuffle.partitions", "3") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print(f"✅ Spark {spark.version} ready!")

✅ Spark 3.5.3 ready!


---
## Cell 2 — Batch Check (Optional)
Verify that Kafka is running and contains messages — this is a simple batch count, not a streaming read.

> Note: The batch read takes an instant snapshot of the topic and does not affect the stream.

In [2]:
df_batch = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "host.docker.internal:9094") \
    .option("subscribe", "real_match_events") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()

print(f"✅ Messages in Kafka: {df_batch.count():,}")
df_batch.selectExpr("CAST(value AS STRING)").show(3, truncate=80)

✅ Messages in Kafka: 0
+-----+
|value|
+-----+
+-----+



---
## Cell 3 — Define the Streaming Source
Configure Spark to read from Kafka as a **continuous stream**.

- `startingOffsets: latest` → Only consume new messages produced after the stream starts
- `maxOffsetsPerTrigger: 1000` → Process at most 1000 messages per micro-batch (every 30 seconds)
- `failOnDataLoss: false` → Do not fail if messages were deleted from Kafka before being consumed

In [3]:
df_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "host.docker.internal:9094") \
    .option("subscribe", "real_match_events") \
    .option("startingOffsets", "latest") \
    .option("maxOffsetsPerTrigger", 1000) \
    .option("failOnDataLoss", "false") \
    .load()

print("✅ Streaming source defined!")
print(f"Schema: {df_stream.schema.simpleString()}")

✅ Streaming source defined!
Schema: struct<key:binary,value:binary,topic:string,partition:int,offset:bigint,timestamp:timestamp,timestampType:int>


---
## Cell 4 — Start the Bronze Stream
This is the core of the Bronze Layer:

1. Cast the Kafka `value` to string (raw JSON)
2. Add 3 metadata columns:
   - `_bronze_ingested_at` → Timestamp when the event arrived at the Bronze layer
   - `_source_topic` → Name of the Kafka topic
   - `_pipeline` → Name of the pipeline
3. Write Parquet files every 30 seconds

> **Important:** The stream keeps running in the background — do not call `awaitTermination` here.

In [4]:
from pyspark.sql.functions import current_timestamp, lit

# Output and checkpoint paths
bronze_path     = os.path.expanduser("~/footballflow/bronze/match_events")
checkpoint_path = os.path.expanduser("~/footballflow/checkpoints/bronze_match_events")

# Create directories if they do not exist
os.makedirs(bronze_path, exist_ok=True)
os.makedirs(checkpoint_path, exist_ok=True)

# Bronze transformation:
# - raw_json            : Full event as received from Kafka (string)
# - _bronze_ingested_at : Timestamp when the event was received in the Bronze layer
# - _source_topic       : Kafka topic name
# - _pipeline           : Pipeline identifier
bronze_stream = df_stream \
    .selectExpr("CAST(value AS STRING) as raw_json") \
    .withColumn("_bronze_ingested_at", current_timestamp()) \
    .withColumn("_source_topic", lit("real_match_events")) \
    .withColumn("_pipeline",     lit("FootballFlow_Bronze"))

# Start the stream and write Parquet files
bronze_query = bronze_stream \
    .writeStream \
    .format("parquet") \
    .option("path",               bronze_path) \
    .option("checkpointLocation", checkpoint_path) \
    .outputMode("append") \
    .trigger(processingTime="30 seconds") \
    .start()

print("✅ Bronze Layer stream started!")
print(f"📁 Output path : {bronze_path}")
print(f"📁 Checkpoint  : {checkpoint_path}")
print(f"🔄 Status      : {'Running' if bronze_query.isActive else 'Stopped'}")

✅ Bronze Layer stream started!
📁 Output path : /home/jovyan/footballflow/bronze/match_events
📁 Checkpoint  : /home/jovyan/footballflow/checkpoints/bronze_match_events
🔄 Status      : Running


---
## Cell 5 — Check Stream Status
Verify that the stream is active and writing data — run this cell 30–60 seconds after starting the GUI and sending events.

In [5]:
import time

print(f"🔄 Stream active : {bronze_query.isActive}")
print(f"📊 Last progress : {bronze_query.lastProgress}")

🔄 Stream active : True
📊 Last progress : None


---
## Cell 6 — Verify Bronze Files
Read the written Parquet files and validate the data.

> Run this cell after the GUI has sent events and at least 30 seconds have passed.

In [8]:
import pandas as pd

bronze_path = os.path.expanduser("~/footballflow/bronze/match_events/")

# Count the existing Parquet files
all_files = [f for f in os.listdir(bronze_path) if f.endswith(".parquet")]
print(f"📁 Parquet files : {len(all_files)}")

# Read the data
df = pd.read_parquet(bronze_path)
print(f"📊 Total rows    : {len(df)}")
print(f"📋 Columns       : {df.columns.tolist()}")
print("\n🔍 Sample data:")
print(df[["raw_json", "_bronze_ingested_at", "_source_topic"]].head(3))

📁 Parquet files : 15
📊 Total rows    : 21
📋 Columns       : ['raw_json', '_bronze_ingested_at', '_source_topic', '_pipeline']

🔍 Sample data:
                                            raw_json     _bronze_ingested_at  \
0  {"game_event_id": "0aa47ccef07d44a80563c0196ed... 2026-06-12 21:16:30.024   
1  {"game_event_id": "50a2d05f6366230ed907e178ec8... 2026-06-11 20:02:30.030   
2  {"game_event_id": "8a2354190d2a927204494809231... 2026-06-11 20:05:00.023   

       _source_topic  
0  real_match_events  
1  real_match_events  
2  real_match_events  


---
## Cell 7 — Stop the Stream
Once you have confirmed the data was written correctly, stop the stream.

In [7]:
# Stop the Bronze stream
bronze_query.stop()
print("⏹ Bronze stream stopped")

⏹ Bronze stream stopped


---
### Utility — Clear all Bronze files (use with caution)


In [7]:
rm -rf ~/footballflow/bronze/match_events/* ~/footballflow/checkpoints/bronze_match_events/*

In [9]:
# Stop all active streams
for s in spark.streams.active:
    s.stop()

In [10]:
# Stop the Spark session
spark.stop()